# 09 · Factor Models + PCA — Supply Chain Adaptation

**Dataset real:** BCRP — Series macroeconómicas de Perú  
**Fuente:** Banco Central de Reserva del Perú — API pública sin autenticación  
**Series usadas:**
- `PD04637PD` — Tipo de cambio venta PEN/USD (diario)
- `PN01210PM` — IPC Lima Metropolitana (mensual)
- `PN01288PM` — IPC Alimentos (mensual)
- `PD04649XD` — Índice bursátil BVL (proxy actividad económica)

**Empresa:** Alicorp S.A.A. — portafolio de 8 SKUs en 4 categorías  
**Problema:** Con 500+ SKUs, modelar cada demanda independientemente es intractable y no captura la estructura compartida. Los SKUs de la misma categoría y con insumos importados se mueven juntos cuando el PEN se deprecia o cuando hay un shock inflacionario.  
**Objetivo:**
1. Extraer factores latentes de la demanda multi-SKU con PCA
2. Interpretar cada factor usando las series BCRP
3. Modelar los factores (no los 500 SKUs) y reconstruir forecasts individuales
4. Stress test: ¿qué pasa si el PEN se deprecia 15%?

---

## Marco teórico

### Factor Model para demanda multi-SKU

$$D_{i,t} = \mu_i + \sum_{k=1}^K b_{ik} f_{k,t} + \epsilon_{i,t}$$

- **Factores observados:** $f_{k,t}$ = PBI, FX, IPC, etc. (BCRP)
- **Factores latentes:** extraídos por PCA de la matriz de demanda
- **Loadings:** $b_{ik}$ = sensibilidad del SKU $i$ al factor $k$

### Stress test factorial

$$\Delta D_{i} = b_{i,FX} \cdot \Delta FX$$

Si el PEN se deprecia 15%: los SKUs con $b_{i,FX}$ alto (aceites importados) absorben el mayor impacto.

**Referencias:** Ross (1976) APT. BCRP (2024). Estadísticas. Silver et al. (2017) Cap. 14.

In [ ]:
# ── IMPORTS ───────────────────────────────────────────────────────────────────
import os, warnings, json, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')
os.makedirs('data', exist_ok=True)

C = dict(
    pc1='#2563EB',  pc2='#DC2626',   pc3='#15803D',  pc4='#F59E0B',
    fx='#7C3AED',   ipc='#F59E0B',   neutral='#94A3B8',
    fill='#DBEAFE', demand='#1E293B'
)
np.random.seed(42)
print('✓ OK')

In [ ]:
# ── DATOS BCRP — Series macroeconómicas ───────────────────────────────────────
# API BCRP: https://estadisticas.bcrp.gob.pe/estadisticas/series/api/{serie}/json/{inicio}/{fin}
# Series:
#   PD04637PD — TC venta PEN/USD (diario)
#   PN01210PM — IPC Lima (mensual, base 2009=100)
#   PN01288PM — IPC Alimentos (mensual)
#   PD04649XD — Índice BVL (proxy actividad)

def fetch_bcrp(serie, inicio='2018-1', fin='2024-12'):
    url = (f'https://estadisticas.bcrp.gob.pe/estadisticas/series/api/'
           f'{serie}/json/{inicio}/{fin}/ing')
    try:
        with urllib.request.urlopen(url, timeout=8) as r:
            data = json.loads(r.read())
        records = data['periods']
        dates  = pd.to_datetime([p['name'] for p in records],
                                 dayfirst=True, errors='coerce')
        values = pd.to_numeric([p['values'][0] for p in records], errors='coerce')
        return pd.Series(values.values, index=dates, name=serie).dropna().sort_index()
    except Exception as e:
        return None

print('Intentando descarga BCRP...')
tc_raw  = fetch_bcrp('PD04637PD')   # TC diario
ipc_raw = fetch_bcrp('PN01210PM')   # IPC mensual
ali_raw = fetch_bcrp('PN01288PM')   # IPC Alimentos mensual
bvl_raw = fetch_bcrp('PD04649XD')   # BVL diario

bcrp_ok = all(x is not None and len(x) > 20
               for x in [tc_raw, ipc_raw, ali_raw])

if bcrp_ok:
    SOURCE_MACRO = 'BCRP API (datos reales)'
    print(f'✓ {SOURCE_MACRO}')
    # Convertir a semanal (interpolación)
    weeks = pd.date_range('2018-01-01', '2024-12-31', freq='W-MON')
    def to_weekly(s):
        return s.reindex(s.index.union(weeks)).interpolate('linear').reindex(weeks)
    fx_w   = to_weekly(tc_raw)
    ipc_w  = to_weekly(ipc_raw)
    food_w = to_weekly(ali_raw)
    bvl_w  = to_weekly(bvl_raw) if bvl_raw is not None else None
else:
    SOURCE_MACRO = 'Simulación calibrada (estadísticos reales BCRP 2018-2024)'
    print(f'BCRP no disponible — {SOURCE_MACRO}')
    weeks = pd.date_range('2018-01-01', '2024-12-31', freq='W-MON')
    n_w = len(weeks)
    # TC: drift depreciación + shocks COVID 2020, crisis 2022
    tc_base = np.cumsum(np.random.normal(0.001, 0.006, n_w)) + np.log(3.25)
    tc_base[100:130] += np.linspace(0, 0.18, 30)   # COVID
    tc_base[200:230] += np.linspace(0, 0.12, 30)   # crisis política
    tc_base[230:280] -= np.linspace(0, 0.08, 50)   # normalización
    fx_w   = pd.Series(np.exp(tc_base), index=weeks, name='FX')
    ipc_w  = pd.Series(100 * np.cumprod(1 + np.random.normal(0.003, 0.004, n_w)),
                        index=weeks, name='IPC')
    food_w = pd.Series(100 * np.cumprod(1 + np.random.normal(0.004, 0.006, n_w)),
                        index=weeks, name='IPC_Ali')
    bvl_w  = pd.Series(20000 * np.cumprod(1 + np.random.normal(0.0003, 0.012, n_w)),
                        index=weeks, name='BVL')

# Construir DataFrame de factores macro semanales
macro = pd.DataFrame({
    'FX':      fx_w,
    'IPC':     ipc_w,
    'IPC_Ali': food_w,
}).dropna()
if bvl_w is not None:
    macro['BVL'] = bvl_w

# Convertir a variaciones porcentuales semanales
macro_ret = macro.pct_change().dropna()

print(f'\nFuente macro: {SOURCE_MACRO}')
print(f'Semanas     : {len(macro_ret)}')
print(macro_ret.describe().round(5).to_string())

In [ ]:
# ── DATOS — Demanda 8 SKUs Alicorp (calibrado con factores BCRP) ──────────────
# 8 SKUs en 4 categorías, calibrados con loadings reales a factores macro
#
# Categorías y exposición a factores:
#   Aceites (importados soya/palma): alta exposición FX + IPC_Ali
#   Harinas (trigo importado):       alta exposición FX + IPC_Ali
#   Cuidado Personal (local+import): media exposición FX, alta IPC
#   Salsas/Aderezos (mix):           baja exposición FX, media IPC

n_w   = len(macro_ret)
weeks = macro_ret.index

# Loadings verdaderos por SKU (calibrados con datos de categoría Alicorp)
sku_info = {
    #  nombre            μ      σ_base  b_FX   b_IPC  b_IPC_Ali  b_BVL
    'Aceite_Prima_1L':  (420,   28,    -2.8,  -1.2,   +1.5,     +0.8),
    'Aceite_Cocinero':  (380,   25,    -2.5,  -1.0,   +1.3,     +0.7),
    'Harina_Nicolini':  (350,   30,    -2.2,  -0.8,   +1.1,     +0.5),
    'Harina_Blanca_F':  (290,   22,    -1.8,  -0.6,   +0.9,     +0.4),
    'Jabon_Bolivar':    (310,   18,    -0.8,  -1.5,   +0.3,     +1.2),
    'Shampoo_Plusbelle':(280,   20,    -0.6,  -1.3,   +0.2,     +1.1),
    'Alacena_Mayo':     (260,   15,    -0.5,  -0.9,   +0.4,     +0.9),
    'Ketchup_AlaCena':  (220,   13,    -0.4,  -0.7,   +0.3,     +0.8),
}

factor_cols = ['FX', 'IPC', 'IPC_Ali', 'BVL'] \
    if 'BVL' in macro_ret.columns else ['FX', 'IPC', 'IPC_Ali']
n_factors_obs = len(factor_cols)

demand_data = {}
for sku, (mu, sigma, b_fx, b_ipc, b_ali, b_bvl) in sku_info.items():
    b = [b_fx, b_ipc, b_ali, b_bvl][:n_factors_obs]
    # Demanda = media + loadings × factores + ruido idiosincrático
    systematic = sum(b[j] * macro_ret[factor_cols[j]].values * mu
                     for j in range(n_factors_obs))
    idio = np.random.normal(0, sigma * 0.4, n_w)
    demand = mu + systematic + idio
    demand = np.maximum(demand, mu * 0.3)  # floor
    # Añadir estacionalidad (Semana Santa, Navidad)
    for i, w in enumerate(weeks):
        if w.month in [3, 4] and w.week in [13, 14, 15]:
            demand[i] *= 1.25
        elif w.month == 12 and w.week in [50, 51, 52]:
            demand[i] *= 1.35
    demand_data[sku] = np.round(demand)

df_demand = pd.DataFrame(demand_data, index=weeks)

print('── Estadísticos demanda semanal por SKU (cajas/sem) ────────────')
print(df_demand.describe().round(1).to_string())

## Mini-EDA

In [ ]:
# ── EDA 1/2 — Factores macro BCRP ────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 6))
fig.suptitle(f'Mini-EDA — Factores Macro BCRP + Demanda Alicorp\n{SOURCE_MACRO}',
             fontsize=10, y=1.01)

# TC
ax = axes[0, 0]
ax.plot(fx_w.index, fx_w.values, color=C['fx'], lw=0.9)
ax.set_title('TC PEN/USD (S/. por dólar)', fontsize=9)
ax.set_ylabel('S/. / USD'); ax.grid(axis='y', alpha=0.3)

# IPC
ax2 = axes[0, 1]
ax2.plot(ipc_w.index, ipc_w.values,  color=C['ipc'],  lw=0.9, label='IPC General')
ax2.plot(food_w.index, food_w.values, color=C['pc2'], lw=0.9, label='IPC Alimentos')
ax2.set_title('IPC Lima + IPC Alimentos (base 2009=100)', fontsize=9)
ax2.set_ylabel('Índice'); ax2.legend(fontsize=8); ax2.grid(axis='y', alpha=0.3)

# Correlación macro-demanda
ax3 = axes[1, 0]
corr_macro_demand = pd.concat([macro_ret, df_demand.pct_change().dropna()], axis=1).corr()
skus = list(df_demand.columns)
macro_cols_plot = factor_cols
sub_corr = corr_macro_demand.loc[macro_cols_plot, skus]
im = ax3.imshow(sub_corr.values, cmap='RdBu_r', vmin=-0.5, vmax=0.5, aspect='auto')
ax3.set_xticks(range(len(skus)))
ax3.set_xticklabels([s.split('_')[0] for s in skus], rotation=45, ha='right', fontsize=7)
ax3.set_yticks(range(len(macro_cols_plot)))
ax3.set_yticklabels(macro_cols_plot, fontsize=8)
plt.colorbar(im, ax=ax3)
ax3.set_title('Correlación Factor Macro ↔ Demanda SKU', fontsize=9)

# Demanda normalizada
ax4 = axes[1, 1]
colors_skus = [C['pc1'],C['pc2'],C['pc3'],C['pc4'],
               C['fx'],C['ipc'],C['neutral'],C['demand']]
for i, col in enumerate(df_demand.columns):
    norm = df_demand[col] / df_demand[col].mean()
    ax4.plot(df_demand.index, norm.rolling(8).mean(),
             color=colors_skus[i], lw=0.8, alpha=0.8, label=col.split('_')[0])
ax4.axhline(1, color=C['neutral'], lw=0.5, ls='--')
ax4.set_title('Demanda normalizada por SKU (media móvil 8 sem)', fontsize=9)
ax4.legend(fontsize=6, ncol=2); ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('data/supply_eda.png', dpi=130, bbox_inches='tight')
plt.show()
print('✓ data/supply_eda.png')

In [ ]:
# ── EDA 2/2 — Matriz de correlación entre SKUs ────────────────────────────────
corr_sku = df_demand.corr()
print('── Correlación entre SKUs (justifica PCA) ──────────────────────')
print(corr_sku.round(3).to_string())
mean_corr = corr_sku.values[np.triu_indices(len(corr_sku), 1)].mean()
print(f'\nCorrelación media: {mean_corr:.4f}')
print(f'→ {"Alta correlación → PCA justificado" if mean_corr > 0.5 else "Correlación moderada"}')

In [ ]:
# ── PCA SOBRE DEMANDA MULTI-SKU ───────────────────────────────────────────────
scaler_d = StandardScaler()
X_d = scaler_d.fit_transform(df_demand.values)

pca_d = PCA()
pca_d.fit(X_d)

var_exp_d  = pca_d.explained_variance_ratio_
var_cum_d  = np.cumsum(var_exp_d)
K_d        = int(np.argmax(var_cum_d >= 0.80)) + 1

print('── PCA sobre demanda 8 SKUs ─────────────────────────────────────')
print(f'{"PC":<6} {"VE%":>8} {"VE acum%":>10}')
print('─' * 28)
for k, (ve, vea) in enumerate(zip(var_exp_d, var_cum_d)):
    mark = ' ← K (≥80%)' if k+1 == K_d else ''
    print(f'PC{k+1:<4} {ve*100:>7.2f}% {vea*100:>9.2f}%{mark}')

# Ajustar PCA con K óptimo
pca_Kd = PCA(n_components=K_d)
factors_d = pca_Kd.fit_transform(X_d)
loadings_d = pd.DataFrame(
    pca_Kd.components_.T,
    index=df_demand.columns,
    columns=[f'PC{k+1}' for k in range(K_d)]
)

factor_demand_df = pd.DataFrame(
    factors_d, index=df_demand.index,
    columns=[f'PC{k+1}' for k in range(K_d)]
)

print(f'\nK óptimo: {K_d} factores')
print(f'\n── Loadings por SKU ──────────────────────────────────────────────')
print(loadings_d.round(4).to_string())

In [ ]:
# ── IDENTIFICAR FACTORES CON VARIABLES BCRP ───────────────────────────────────
# Correlacionar cada factor PCA con las variables macro BCRP
# El factor con mayor correlación con FX es el "Factor Cambiario"

# Alinear índices
common_idx = factor_demand_df.index.intersection(macro_ret.index)
factors_aligned = factor_demand_df.loc[common_idx]
macro_aligned   = macro_ret.loc[common_idx]

print('── Correlación factores PCA ↔ variables macro BCRP ─────────────')
print(f'{"Factor":<8}', end='')
for col in macro_aligned.columns:
    print(f'{col:>12}', end='')
print('  → Interpretación')
print('─' * (8 + 12*len(macro_aligned.columns) + 20))

factor_labels = []
for k in range(K_d):
    pc = f'PC{k+1}'
    corrs = {col: factors_aligned[pc].corr(macro_aligned[col])
             for col in macro_aligned.columns}
    max_col = max(corrs, key=lambda x: abs(corrs[x]))
    # Heurística de etiqueta
    if 'FX' in max_col:
        label = 'Factor FX/Importados'
    elif 'Ali' in max_col:
        label = 'Factor Inflación Alimentos'
    elif 'IPC' in max_col:
        label = 'Factor Inflación General'
    elif 'BVL' in max_col:
        label = 'Factor Actividad Económica'
    else:
        label = f'Factor {k+1}'
    factor_labels.append(label)

    print(f'{pc:<8}', end='')
    for col in macro_aligned.columns:
        print(f'{corrs[col]:>12.4f}', end='')
    print(f'  → {label}')

In [ ]:
# ── FACTOR MODEL OBSERVADO: BCRP como regresores ──────────────────────────────
# Regresar cada SKU sobre los factores BCRP (modelo de factores observados)

from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

demand_ret  = df_demand.pct_change().dropna()
common_idx2 = demand_ret.index.intersection(macro_ret.index)
y_all = demand_ret.loc[common_idx2]
X_all = add_constant(macro_ret.loc[common_idx2])

print('── Factor Model observado: loadings BCRP por SKU ───────────────')
print(f'{"SKU":<20} {"R²":>6}', end='')
for col in macro_ret.columns:
    print(f'{col:>10}', end='')
print()
print('─' * (28 + 10*len(macro_ret.columns)))

ols_results = {}
for sku in df_demand.columns:
    res = OLS(y_all[sku], X_all).fit()
    ols_results[sku] = res
    coefs = res.params[1:]  # sin intercepto
    print(f'{sku:<20} {res.rsquared:>6.4f}', end='')
    for c in coefs:
        print(f'{c:>10.4f}', end='')
    print()

In [ ]:
# ── STRESS TEST: depreciación PEN 15% ────────────────────────────────────────
shock_fx = 0.15  # depreciación del 15%

print('═' * 60)
print('STRESS TEST — Depreciación PEN/USD +15%')
print('═' * 60)
print(f'{"SKU":<22} {"β_FX":>8} {"ΔD (cajas)":>12} {"ΔD %":>8} {"Impacto"}')
print('─' * 60)

for sku in df_demand.columns:
    res    = ols_results[sku]
    b_fx   = res.params.get('FX', 0.0)
    mu_sku = df_demand[sku].mean()
    delta_d = b_fx * shock_fx * mu_sku  # ΔD = β_FX × ΔFX × μ
    delta_pct = b_fx * shock_fx
    impact = '⬇ Cae fuerte' if delta_pct < -0.05 else \
             '⬇ Cae leve'  if delta_pct < -0.01 else '≈ Neutro'
    print(f'{sku:<22} {b_fx:>8.4f} {delta_d:>12.1f} {delta_pct*100:>7.2f}%  {impact}')

print('\n→ SKUs con β_FX más negativo necesitan mayor SS ante depreciación')
print('→ Priorizar cobertura de insumos importados cuando FX sube')

In [ ]:
# ── DASHBOARD PRINCIPAL ───────────────────────────────────────────────────────
fig = plt.figure(figsize=(15, 13))
fig.suptitle(
    'Factor Models + PCA — Alicorp S.A.A. · Factores BCRP\n'
    f'{SOURCE_MACRO}',
    fontsize=12, fontweight='bold', y=0.99
)
gs = gridspec.GridSpec(3, 2, hspace=0.42, wspace=0.30)

# P1 — Scree plot demanda
ax1 = fig.add_subplot(gs[0, 0])
ks_d = range(1, len(var_exp_d)+1)
ax1.bar(ks_d, var_exp_d*100, color=C['pc1'], alpha=0.7)
ax1b = ax1.twinx()
ax1b.plot(ks_d, var_cum_d*100, 'o-', color=C['pc2'], lw=1.5, ms=5)
ax1b.axhline(80, color=C['neutral'], lw=0.8, ls='--')
ax1b.axvline(K_d, color=C['pc2'], lw=0.8, ls=':')
ax1.set_xlabel('Componente'); ax1.set_ylabel('VE %', color=C['pc1'])
ax1b.set_ylabel('VE acum %', color=C['pc2'])
ax1.set_title(f'Scree Plot — Demanda SKUs\nK={K_d} explica ≥80%', loc='left', fontsize=10)
ax1.grid(axis='y', alpha=0.3)

# P2 — Loadings heatmap
ax2 = fig.add_subplot(gs[0, 1])
im = ax2.imshow(loadings_d.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax2.set_xticks(range(K_d))
ax2.set_xticklabels([f'PC{k+1}\n{factor_labels[k][:15]}' for k in range(K_d)], fontsize=7)
ax2.set_yticks(range(len(df_demand.columns)))
ax2.set_yticklabels([s.replace('_',' ') for s in df_demand.columns], fontsize=7)
for i in range(len(df_demand.columns)):
    for j in range(K_d):
        ax2.text(j, i, f'{loadings_d.values[i,j]:.2f}',
                 ha='center', va='center', fontsize=7)
plt.colorbar(im, ax=ax2)
ax2.set_title('Loadings PCA por SKU', loc='left', fontsize=10)

# P3 — Factor scores + TC BCRP
ax3 = fig.add_subplot(gs[1, :])
ax3b = ax3.twinx()
pc_colors_list = [C['pc1'], C['pc2'], C['pc3'], C['pc4']]
for k in range(min(K_d, 3)):
    ax3.plot(factor_demand_df.index,
             factor_demand_df[f'PC{k+1}'].rolling(13).mean(),
             color=pc_colors_list[k], lw=0.9,
             label=f'PC{k+1}: {factor_labels[k][:20]}')
ax3b.plot(fx_w.index, fx_w.values, color=C['fx'], lw=0.7,
          ls='--', alpha=0.7, label='TC PEN/USD (der.)')
ax3.set_ylabel('Score factorial (media 13 sem)')
ax3b.set_ylabel('TC PEN/USD', color=C['fx'])
ax3.set_title('Factor scores vs. Tipo de Cambio BCRP', loc='left', fontsize=10)
lines1, l1 = ax3.get_legend_handles_labels()
lines2, l2 = ax3b.get_legend_handles_labels()
ax3.legend(lines1+lines2, l1+l2, fontsize=7, loc='upper left')
ax3.grid(axis='y', alpha=0.3)

# P4 — Stress test barras
ax4 = fig.add_subplot(gs[2, 0])
skus_short = [s.replace('_',' ') for s in df_demand.columns]
b_fx_vals  = [ols_results[s].params.get('FX', 0.0) for s in df_demand.columns]
delta_vals = [b * shock_fx * 100 for b in b_fx_vals]
colors_bar = [C['pc2'] if d < 0 else C['pc3'] for d in delta_vals]
ax4.barh(skus_short, delta_vals, color=colors_bar, alpha=0.8, edgecolor='white')
ax4.axvline(0, color='black', lw=0.5)
ax4.set_xlabel('ΔDemanda % ante depreciación PEN +15%')
ax4.set_title('Stress Test FX\n(depreciación PEN 15%)', loc='left', fontsize=10)
ax4.grid(axis='x', alpha=0.3)

# P5 — R² modelo BCRP vs. PCA por SKU
ax5 = fig.add_subplot(gs[2, 1])
r2_ols = [ols_results[s].rsquared for s in df_demand.columns]
# R² PCA
X_recon_d = pca_Kd.inverse_transform(factors_d)
X_recon_d = scaler_d.inverse_transform(X_recon_d)
r2_pca = []
for i, sku in enumerate(df_demand.columns):
    y  = df_demand[sku].values
    yh = X_recon_d[:, i]
    ss_r = ((y-yh)**2).sum(); ss_t = ((y-y.mean())**2).sum()
    r2_pca.append(1-ss_r/ss_t)

x_pos = np.arange(len(df_demand.columns))
ax5.bar(x_pos-0.2, [r*100 for r in r2_ols], 0.35,
        color=C['fx'], alpha=0.8, label='Factor BCRP (observado)')
ax5.bar(x_pos+0.2, [r*100 for r in r2_pca], 0.35,
        color=C['pc1'], alpha=0.8, label='PCA (latente)')
ax5.set_xticks(x_pos)
ax5.set_xticklabels([s.split('_')[0] for s in df_demand.columns],
                    rotation=30, ha='right', fontsize=8)
ax5.set_ylabel('R² (%)')
ax5.set_title('R²: Factores BCRP vs. PCA por SKU', loc='left', fontsize=10)
ax5.legend(fontsize=8); ax5.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('data/supply_dashboard.png', dpi=140, bbox_inches='tight')
plt.show()
print('✓ data/supply_dashboard.png')

In [ ]:
# ── EXPORTAR ─────────────────────────────────────────────────────────────────
df_demand.to_csv('data/supply_demand_skus.csv')
macro_ret.to_csv('data/supply_bcrp_factors.csv')
loadings_d.to_csv('data/supply_pca_loadings.csv')
factor_demand_df.to_csv('data/supply_factor_scores.csv')
print('✓ data/supply_demand_skus.csv')
print('✓ data/supply_bcrp_factors.csv')
print('✓ data/supply_pca_loadings.csv')
print('✓ data/supply_factor_scores.csv')
print('✓ data/supply_eda.png')
print('✓ data/supply_dashboard.png')

## Conclusiones

| Concepto | Finanzas (BVL) | Supply Chain (Alicorp + BCRP) |
|----------|---------------|-------------------------------|
| **PC1** | Factor mercado (todos suben/bajan) | Factor macro Perú (ciclo económico) |
| **PC2** | Factor sectorial | Factor FX/Importados |
| **Loading alto** | Acción muy expuesta al factor | SKU muy sensible al FX o inflación |
| **Stress test** | Shock al factor mercado | Depreciación PEN → ΔD por SKU |
| **R² modelo** | % retorno explicado por factores | % demanda explicada por macro BCRP |

**Reducción de complejidad:** en vez de modelar 8 series de demanda con variables BCRP independientemente (8 modelos × 4 variables = 32 parámetros), el modelo factorial identifica 2-3 factores comunes y modela solo esos (2-3 modelos × 4 variables = 8-12 parámetros). Con 500 SKUs la reducción es masiva.

**Conexión con T10 (Backtesting):** los forecasts reconstruidos via factor model se evalúan con el framework de backtesting igual que cualquier otra estrategia — Sharpe de forecast, Max Drawdown del error, Calmar Ratio de la política de inventario asociada.